In [15]:
import json
from collections import defaultdict



PDNS_NAMES = [
    "cira",
    "cloudflare",
    "cisco",
    "quad9"
    ]

detections_by_domain = defaultdict(set)  # domain -> set of PDNS names that detected it
detections_by_pdns = defaultdict(set)  # PDNS name -> set of domains detected by that PDNS

def is_blocked_by_cisco(a_data):
    """Check if the A record answers indicate a block by Cisco Umbrella.

    Cisco Umbrella does not return Extended DNS Error (EDE) codes for
    blocked domains like Cloudflare, CIRA, and Quad9 do.

    Instead, it returns an IP address (146.112.61.108) for blocked domains, which
    changes depending on the block type (malware, phishing, adult, etc.).
    https://securitydocs.cisco.com/docs/umbrella-dns/olh/146809.dita

    We only use the malware and phishing block IP address, which was
    obtained by querying Cisco's 2 test domains:
    examplemalwaredomain.com (Malware), internetbadguys.com (Phishing).

    Can be verified or updated by running:
    dig +short A internetbadguys.com @208.67.222.222
    dig +short A examplemalwaredomain.com @208.67.222.222"""

    CISCO_UMBRELLA_BLOCKED_IPS = ["146.112.61.108"]  # malware and phishing block IP address
    answers = a_data.get("answers", [])
    for answer in answers:
        if answer.get("answer") in CISCO_UMBRELLA_BLOCKED_IPS:
            return True
    return False


def is_blocked_by_ede(additionals):
    """Check if the additional records indicate a block by EDE codes."""
    for additional in additionals:
        ede_list = additional.get("ede", [])
        for ede in ede_list:
            info_code = ede.get("info_code")
            if info_code in [16, 17]:  # 16 - Censored. 17 - Filtered.
                return True
    return False


for pdns_name in PDNS_NAMES:
    with open(f"pdns_{pdns_name}_ad_links.jsonl", "r") as in_f:
        for line in in_f:
            data = json.loads(line)
            domain = data["name"]
            results = data["results"]
            if "A" not in results:
                continue
            a_data = results["A"]["data"]
            additionals = a_data.get("additionals", [])
            if is_blocked_by_cisco(a_data) or is_blocked_by_ede(additionals):
                detections_by_domain[domain].add(pdns_name)
                detections_by_pdns[pdns_name].add(domain)

# print summary of detections, sorted by number of PDNS that detected each domain
for domain, pdns_set in sorted(detections_by_domain.items(), key=lambda x: len(x[1]), reverse=True):
    if len(pdns_set) > 1:
        print(f"{domain}: detected by {len(pdns_set)} PDNS ({', '.join(sorted(pdns_set))})")

# for pdns_name, domains in sorted(detections_by_pdns.items(), key=lambda x: len(x[1])):
#     print(f"{pdns_name}: detected {len(domains)} domains, {', '.join(sorted(domains))}")

spolecznosclokal.convertri.com: detected by 3 PDNS (cira, cloudflare, quad9)
wp.iman-pl.com: detected by 3 PDNS (cira, cisco, quad9)
Licenzegenius.it: detected by 2 PDNS (cira, cloudflare)
eu.yourfavouritedocs.com: detected by 2 PDNS (cira, cloudflare)
gopdfmanuals.com: detected by 2 PDNS (cira, cloudflare)
pl.getniy.shop: detected by 2 PDNS (cira, cloudflare)
pdfscraper.com: detected by 2 PDNS (cira, cloudflare)
Keysoft.store: detected by 2 PDNS (cira, cloudflare)
Dropland.net: detected by 2 PDNS (cira, cloudflare)
kajoyefoqumanalytics.click: detected by 2 PDNS (cira, cloudflare)
pyproxy.com: detected by 2 PDNS (cira, quad9)
www.nerdused.com: detected by 2 PDNS (cira, cloudflare)
